# Feature Extraction for VSX → TESS Variable-Star Classification

This notebook extracts interpretable machine-learning features from standardized TESS light-curve FITS files.

It is designed for the current research pipeline:

- **Input**: metadata parquet containing `lightCurvePath`
- **Light curve source**: standardized FITS files already saved by `TessDataDownloader.py`
- **Output**: feature parquet, one row per star
- **Parallel processing**: configurable thread pool, default `WorkerCount = 16`

Important leakage rule:

> Feature extraction here is safe before train/validation/test split because every feature is computed independently per star.  
> Dataset-level scaling, PCA, feature selection, and model fitting must happen only after train/validation/test split.

# Complete Feature List

The actual output parquet contains **58 columns**. The list below matches the schema of:

```text
TESS_features_2026_07_29_v1_input.parquet
```

## 1. Identification and Metadata Columns

| Column | Meaning | Method / Source |
|---|---|---|
| `family` | VSX family used as the classification target | copied from input metadata |
| `VSXType` | original or mapped VSX variability type | copied from input metadata |
| `VSXId` | VSX identifier | copied from input metadata |
| `lightCurvePath` | standardized or conditionally detrended FITS path used by feature extraction | copied from input metadata |
| `rawLightCurvePath` | original raw FITS path | copied from input metadata |
| `QualityLabel` | resolved light-curve quality label | QualityLabel summarizes the fraction of cadences rejected by the TESS quality mask: clean (<10%), acceptable (10–<25%), caution (25–<40%), poor (≥40%), or missing when no usable light curve was acquired. QualityScore is the numeric encoding of those labels: 0, 1, 2, 3, and 4 respectively. A higher score therefore indicates worse or unavailable data quality. It is an ordinal pipeline-quality indicator, not a physical measurement of the star. TESS quality mask is a set of quality flags assigned by the TESS mission to identify observations that may be unreliable because of spacecraft or instrument effects, such as momentum dumps, scattered light, cosmic rays, or other anomalies. These flagged cadences are typically excluded from scientific analysis. QualityLabel summarizes the fraction of cadences removed by this mask, providing an overall indicator of the reliability of the light curve. |
| `QualityScore` | numeric encoding of `QualityLabel` | `clean=0`, `acceptable=1`, `caution=2`, `poor=3`, `missing=4` |
| `Provenance` | source of the light curve | obtained from `provenance`, then `author`; otherwise `missing` |
| `ProvenanceScore` | numeric encoding of provenance | `SPOC=0`, `QLP=1`, `TESSCut=2`; unrecognized values become `NaN` |
| `OriginalFluxMedian` | median flux before standardization or detrending | Median flux of the original downloaded light curve before standardization or detrending. |
| `OriginalFluxStd` | flux standard deviation before standardization or detrending | Flux standard deviation of the original downloaded light curve before standardization or detrending. |
| `OriginalFluxSnr` | original signal-to-noise estimate |  * See note below |
| `LowSnr` | low-SNR indicator | boolean conversion of input metadata field `lowSNR` |
| `LowQualityLightCurve` | low-quality-light-curve indicator | diagnostic flag set to true when: medianUnstable (median flux is invalid or nearly zero). lowStd (flux standard deviation is nearly zero) lowSNR (abs(medianFlux) / stdFlux < 3)

Note:
* Boolean quality-control flag indicating whether the original downloaded light curve exhibits significant quality issues, such as low signal-to-noise ratio or an excessive fraction of cadences rejected by the TESS quality mask. Light curves flagged as `True` should be interpreted with additional caution during subsequent analysis.

**OriginalFluxSnr** — Estimated signal-to-noise ratio (SNR) of the original downloaded light curve before standardization or detrending, calculated as:

$$
\text{FluxSnr} = \frac{|\mathrm{median}(\mathrm{Flux})|}{\mathrm{std}(\mathrm{Flux})}
$$

A larger value indicates that the signal is stronger relative to the background scatter. This metric is used during data-quality control to identify low-SNR light curves (`LowSnr` is set when `FluxSnr < 3.0`).

## 2. Data Coverage and Cadence Features

| Column | Meaning | Method |
|---|---|---|
| `CadenceCount` | number of retained finite time-and-flux observations | `len(Flux)` after loading and finite-value filtering |
| `TimeSpanDays` | total retained observing baseline in days | `max(Time) - min(Time)` |
| `MedianCadenceDays` | median spacing between adjacent retained observations | `median(diff(Time))` |

## 3. Flux Distribution Features

| Column | Meaning | Method |
|---|---|---|
| `FluxStd` | population standard deviation of retained flux | `std(Flux)` |
| `FluxMedian` | median retained flux | 50th percentile |
| `FluxMad` | median absolute deviation from the median | `median(abs(Flux - FluxMedian))` |
| `FluxP05` | 5th percentile | `percentile(Flux, 5)` |
| `FluxP10` | 10th percentile | `percentile(Flux, 10)` |
| `FluxP90` | 90th percentile | `percentile(Flux, 90)` |
| `FluxP95` | 95th percentile | `percentile(Flux, 95)` |
| `FluxIqr` | interquartile range | `FluxP75 - FluxP25` |
| `FluxAmplitude` | full observed flux range | `FluxMax - FluxMin` |
| `FluxPercentAmplitude95To5` | robust 5th-to-95th percentile amplitude | `FluxP95 - FluxP05` |
| `FluxPercentAmplitude90To10` | robust 10th-to-90th percentile amplitude | `FluxP90 - FluxP10` |
| `FluxSkewness` | bias-corrected asymmetry of the flux distribution | `scipy.stats.skew(Flux, bias=False)` |
| `FluxKurtosis` | bias-corrected excess kurtosis of the flux distribution | `scipy.stats.kurtosis(Flux, bias=False)` |

`FluxSkewness` becomes `NaN` when there are fewer than three retained observations or the flux scatter is effectively zero. `FluxKurtosis` becomes `NaN` when there are fewer than four retained observations or the flux scatter is effectively zero.

## 4. Tail-Asymmetry Features

| Column | Meaning | Method |
|---|---|---|
| `TailUpper` | distance from the median to the 95th percentile | `FluxP95 - FluxMedian` |
| `TailLower` | distance from the 5th percentile to the median | `FluxMedian - FluxP05` |
| `TailAsymmetry` | difference between upper- and lower-tail strengths | `TailUpper - TailLower` |
| `TailRatio` | upper-tail strength relative to lower-tail strength | `TailUpper / (TailLower + Eps)` through safe division. Eps = 1e-12 |

A positive `TailAsymmetry` indicates a stronger upper flux tail. A negative value indicates a stronger lower flux tail.

## 5. Time-Domain Variability Features

| Column | Meaning | Method |
|---|---|---|
| `EtaVonNeumann` | point-to-point variation relative to total variance | `sum(diff(Flux)^2) / ((N - 1) * var(Flux))`. See Notes |
| `MaxAbsSlope` | largest absolute finite rate of flux change | `max(abs(diff(Flux) / diff(Time)))`, excluding invalid or zero time differences |
| `MedianAbsSuccessiveDiff` | typical absolute point-to-point flux change | `median(abs(diff(Flux)))` |
| `FractionBeyond1Std` | fraction of observations more than one standard deviation from the mean | `mean(abs(Flux - FluxMean) > FluxStd)` |
| `FractionBeyond2Std` | fraction of observations more than two standard deviations from the mean | `mean(abs(Flux - FluxMean) > 2 * FluxStd)` |

Notes:
**EtaVonNeumann** — Measures the smoothness of a light curve by comparing point-to-point flux changes with the overall variance. Smaller values indicate smoother, more correlated variability, while larger values indicate noisier or more rapidly changing light curves.

Reference: Kim, D.-W. & Bailer-Jones, C. A. L. (2016), *A package for the automated classification of periodic variable stars*, A&A, 587, A18. https://www.aanda.org/articles/aa/full_html/2016/03/aa27395-15/aa27395-15.html

## 6. Lomb-Scargle Period Features

https://doi.org/10.3847/1538-4365/aab766
https://arxiv.org/abs/1703.09824

The period search is constrained by:

- minimum period: `MinPeriodDays`
- effective maximum period: `min(MaxPeriodDays, 0.9 * TimeSpanDays)`
- grid density: `SamplesPerPeak`

Lomb-Scargle uses flux uncertainty weights only when a valid, positive `FluxErr` array is available. Otherwise, it runs unweighted.

| Column | Meaning | Method |
|---|---|---|
| `LsBestPeriod` | period corresponding to the largest periodogram power | `1 / LsBestFrequency` |
| `LsMaxPower` | highest Lomb-Scargle power | `max(Power)` |
| `LsFalseAlarmProbability` | false-alarm probability of the highest sampled power | `LombScargle.false_alarm_probability(LsMaxPower)` |
| `LsPeriod2` | period corresponding to the second-highest sampled power | inverse of the second-ranked frequency |
| `LsPeriod3` | period corresponding to the third-highest sampled power | inverse of the third-ranked frequency |
| `LsPower2` | second-highest sampled periodogram power | second-ranked value of `Power` |
| `LsPower3` | third-highest sampled periodogram power | third-ranked value of `Power` |
| `LsPowerRatio21` | second-highest power relative to the highest power | `LsPower2 / LsMaxPower` |
| `LsPowerRatio31` | third-highest power relative to the highest power | `LsPower3 / LsMaxPower` |
| `LsPeriodRatio21` | second-ranked period relative to the best period | `LsPeriod2 / LsBestPeriod` |
| `LsPeriodRatio31` | third-ranked period relative to the best period | `LsPeriod3 / LsBestPeriod` |

`LsPeriod2` and `LsPeriod3` currently represent the second- and third-highest sampled frequency-grid powers. The implementation does not require them to be separate independent local peaks.

## 7. Phase-Folded Morphology Features

VanderPlas, J. T. (2018). Understanding the Lomb–Scargle Periodogram. The Astrophysical Journal Supplement Series, 236(1), 16.
https://doi.org/10.3847/1538-4365/aab766

NASA TESS tutorial: https://heasarc.gsfc.nasa.gov/docs/tess/LightCurve-object-Tutorial.html

After identifying the dominant variability period using the Lomb–Scargle periodogram, each light curve is phase-folded by mapping every observation onto a single normalized cycle, where the phase ranges from 0 to 1 (representing one complete period). This aligns repeated cycles and allows their shapes to be directly compared regardless of the original observation times. Phase folding aligns repeated variability cycles, allowing stars with different observation times and periods to be compared using a common representation. Statistical descriptors are then extracted from the folded light curve to characterize its morphology independently of the absolute observation time.

The extracted morphology features summarize complementary physical aspects of the periodic brightness variation. `PhaseCurveRange` and `PhaseCurveStd` quantify the amplitude and overall variability of the folded light curve. `PhaseCurveSmoothness` measures the continuity of the phased profile, helping distinguish smoothly varying pulsators from stars with sharper brightness transitions. `PhasePeakPhase` and `PhaseTroughPhase` identify the phase locations of maximum and minimum brightness, while `PhasePeakToTroughPhaseDelta` measures the phase separation between them, providing a compact description of the light-curve shape and symmetry. Together, these features capture the characteristic morphology of periodic variable stars and complement the Lomb–Scargle period and power features by describing **how** the brightness changes during each cycle rather than **how often** it repeats.

The light curve is folded using `LsBestPeriod`, divided into `PhaseBinCount` bins, and summarized using the median flux in each nonempty bin. At least five nonempty bins are required.

| Column | Meaning | Method |
|---|---|---|
| `PhaseCurveStd` | standard deviation of the median-binned phase curve | `std(BinnedFlux)` |
| `PhaseCurveRange` | full range of the median-binned phase curve | `max(BinnedFlux) - min(BinnedFlux)` |
| `PhaseCurveSmoothness` | scatter of cyclic differences between adjacent phase bins | standard deviation of cyclic first differences |
| `PhasePeakPhase` | median phase coordinate of the maximum-flux bin | `BinnedPhase[argmax(BinnedFlux)]` |
| `PhaseTroughPhase` | median phase coordinate of the minimum-flux bin | `BinnedPhase[argmin(BinnedFlux)]` |
| `PhasePeakToTroughPhaseDelta` | shortest cyclic phase separation between peak and trough | `min(abs(PeakPhase - TroughPhase), 1 - abs(PeakPhase - TroughPhase))` |

## 8. Extraction Status Columns

| Column | Meaning | Method / Source |
|---|---|---|
| `FeatureStatus` | final outcome of feature extraction for the row | possible values include `ok`, `missing_lightcurve_path`, `missing_lightcurve_file`, `too_few_cadences`, and `failed` |
| `FeatureError` | diagnostic explanation when extraction is not successful | missing path, missing file, retained-cadence count, or exception text |

## Verified Column Count

| Category | Columns |
|---|---:|
| Identification and metadata | 14 |
| Coverage and cadence | 3 |
| Flux distribution | 13 |
| Tail asymmetry | 4 |
| Time-domain variability | 5 |
| Lomb-Scargle | 11 |
| Phase-folded morphology | 6 |
| Extraction status | 2 |
| **Actual total** | **58** |

The following fields from the earlier documentation are **not present** in the actual parquet schema and have therefore been removed:

```text
Name
ticId
bestTicId
ticDistanceArcmin
trendFlag
trendScore
adfPValue
```

## Feature Redundancy Reduction

To improve the stability and interpretability of feature-importance analysis, the extractor no longer outputs:

```text
FluxVariance
FluxMean
FluxP25
FluxP75
FluxMin
FluxMax
FluxP01
FluxP99
```

The retained features preserve the main scientific information: `FluxStd` for standard dispersion, `FluxMedian` for central location, `FluxIqr` for robust spread, `FluxAmplitude` for the full range, and `FluxP05`/`FluxP95` plus the tail features for robust amplitude and asymmetry. The expected output schema is reduced from 67 to **58 columns**.

## Additional Redundancy Reduction

`LsBestFrequency` was removed because it is an exact one-to-one transformation of `LsBestPeriod`:

\[
\mathrm{LsBestFrequency} = \frac{1}{\mathrm{LsBestPeriod}}
\]

`LsBestPeriod` is retained because period in days is more interpretable and is the standard astronomical representation for variable-star behavior. `LowQualityLightCurve` remains in the output feature table as requested. The expected output schema now contains **58 columns**.


# Imports and Configuration

In [1]:
from __future__ import annotations

import logging
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import numpy as np
import pandas as pd

import lightkurve as lk
from astropy.io import fits
from astropy.timeseries import LombScargle
from scipy.stats import skew, kurtosis

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s:%(name)s:%(message)s",
)

Logger = logging.getLogger("FeatureExtractorNotebook")

# Update these paths for your local repo.
# InputMetadataPath = Path("../trend_detection/TESSAugmented_QC_trend.parquet")
# InputMetadataPath = Path("/data/projects/TESS-research/data_pipeline/TESSCache/TESSAugmented_QC.parquet")
# OutputFeaturePath = Path("/data/projects/TESS-research/feature_extraction/TESS_features.parquet")
# InputMetadataPath = Path("/data/projects/TESS-research/detrend_tesscut/output/TESSAugmented_QC_tesscut_conditional_detrended.parquet")
# OutputFeaturePath = Path("/data/projects/TESS-research/detrend_tesscut/output/TESS_conditional_detrended_features.parquet")

MinPeriodDays = 0.05
MaxPeriodDays = 100.0
SamplesPerPeak = 10
MinCadences = 100
PhaseBinCount = 20
WorkerCount = 16
Eps = 1e-12

QLP_TESSCUT_COMPARISON_INPUT_QC='/data/projects/TESS-research/data_pipeline/QLP_TESSCutCache/QLP_TESSCut_Metadata_Consolidated_QC.parquet'
QLP_TESSCUT_COMPARISON_OUTPUT_QC='/data/projects/TESS-research/data_pipeline/QLP_TESSCutCache/QLP_TESSCut_features_QC.parquet'
QLP_TESSCUT_COMPARISON_INPUT_DETREND='/data/projects/TESS-research/data_pipeline/QLP_TESSCutCache/QLP_TESSCut_Metadata_Consolidated_Detrended.parquet'
QLP_TESSCUT_COMPARISON_OUTPUT_DETREND='/data/projects/TESS-research/data_pipeline/QLP_TESSCutCache/QLP_TESSCut_features_Detrended.parquet'

SPOC_TESSCUT_COMPARISON_INPUT_QC='/data/projects/TESS-research/data_pipeline/SPOC_TESSCutCache/SPOC_TESSCut_Metadata_Consolidated_QC.parquet'
SPOC_TESSCUT_COMPARISON_OUTPUT_QC='/data/projects/TESS-research/data_pipeline/SPOC_TESSCutCache/SPOC_TESSCut_features_QC.parquet'
SPOC_TESSCUT_COMPARISON_INPUT_DETREND='/data/projects/TESS-research/data_pipeline/SPOC_TESSCutCache/SPOC_TESSCut_Metadata_Consolidated_Detrended.parquet'
SPOC_TESSCUT_COMPARISON_OUTPUT_DETREND='/data/projects/TESS-research/data_pipeline/SPOC_TESSCutCache/SPOC_TESSCut_features_Detrended.parquet'

QualityScoreMap = {
    "clean": 0,
    "acceptable": 1,
    "caution": 2,
    "poor": 3,
    "missing": 4,
}

ProvenanceScoreMap = {
    "SPOC": 0,
    "QLP": 1,
    "TESSCut": 2,
}

FeatureExtractionCases = {
    "NoDetrend": {
        "InputMetadataPath": Path("/data/projects/TESS-research/data_pipeline/TESSCache/TESSAugmented_QC.parquet"),
        "OutputFeaturePath": Path("/data/projects/TESS-research/feature_extraction/TESS_features.parquet"),
    },
    "TesscutConditionalDetrend": {
        "InputMetadataPath": Path("/data/projects/TESS-research/detrend_tesscut/output/TESSAugmented_QC_tesscut_conditional_detrended.parquet"),
        "OutputFeaturePath": Path("/data/projects/TESS-research/feature_extraction/TESS_conditional_detrended_features.parquet"),
    },
    "SpocTesscutComparisonQC": {
        "InputMetadataPath": Path(SPOC_TESSCUT_COMPARISON_INPUT_QC),
        "OutputFeaturePath": Path(SPOC_TESSCUT_COMPARISON_OUTPUT_QC),
    },
    "QlpTesscutComparisonQC": {
        "InputMetadataPath": Path(QLP_TESSCUT_COMPARISON_INPUT_QC),
        "OutputFeaturePath": Path(QLP_TESSCUT_COMPARISON_OUTPUT_QC),
    },
    "SpocTesscutComparisonDetrend": {
        "InputMetadataPath": Path(SPOC_TESSCUT_COMPARISON_INPUT_DETREND),
        "OutputFeaturePath": Path(SPOC_TESSCUT_COMPARISON_OUTPUT_DETREND),
    },
    "QlpTesscutComparisonDetrend": {
        "InputMetadataPath": Path(QLP_TESSCUT_COMPARISON_INPUT_DETREND),
        "OutputFeaturePath": Path(QLP_TESSCUT_COMPARISON_OUTPUT_DETREND),
    },
}

FeatureDfByCase = {}

for CaseName, Paths in FeatureExtractionCases.items():
    print(f"{CaseName}:")
    print(f"  InputMetadataPath = {Paths['InputMetadataPath']}")
    print(f"  OutputFeaturePath = {Paths['OutputFeaturePath']}")


NoDetrend:
  InputMetadataPath = /data/projects/TESS-research/data_pipeline/TESSCache/TESSAugmented_QC.parquet
  OutputFeaturePath = /data/projects/TESS-research/feature_extraction/TESS_features.parquet
TesscutConditionalDetrend:
  InputMetadataPath = /data/projects/TESS-research/detrend_tesscut/output/TESSAugmented_QC_tesscut_conditional_detrended.parquet
  OutputFeaturePath = /data/projects/TESS-research/feature_extraction/TESS_conditional_detrended_features.parquet
SpocTesscutComparisonQC:
  InputMetadataPath = /data/projects/TESS-research/data_pipeline/SPOC_TESSCutCache/SPOC_TESSCut_Metadata_Consolidated_QC.parquet
  OutputFeaturePath = /data/projects/TESS-research/data_pipeline/SPOC_TESSCutCache/SPOC_TESSCut_features_QC.parquet
QlpTesscutComparisonQC:
  InputMetadataPath = /data/projects/TESS-research/data_pipeline/QLP_TESSCutCache/QLP_TESSCut_Metadata_Consolidated_QC.parquet
  OutputFeaturePath = /data/projects/TESS-research/data_pipeline/QLP_TESSCutCache/QLP_TESSCut_features_QC.

### Comparison-path environment variables

Before running this notebook, define `SPOC_TESSCUT_COMPARISON_INPUT`, `SPOC_TESSCUT_COMPARISON_OUTPUT`, `QLP_TESSCUT_COMPARISON_INPUT`, and `QLP_TESSCUT_COMPARISON_OUTPUT` as full parquet-file paths. The shared extraction loop and final summary will process and display all four configured cases.


# Utility Functions

In [2]:
def SafeFloat(Value: Any) -> float:
    try:
        FloatValue = float(Value)
    except Exception:
        return np.nan
    return FloatValue if np.isfinite(FloatValue) else np.nan


def SafeBool(Value: Any) -> bool:
    if Value is None:
        return False
    if isinstance(Value, float) and pd.isna(Value):
        return False
    return bool(Value)


def SafeDivide(Numerator: float, Denominator: float, Eps: float = Eps) -> float:
    if not np.isfinite(Numerator) or not np.isfinite(Denominator) or abs(Denominator) < Eps:
        return np.nan
    return float(Numerator / Denominator)


def ResolvePath(PathValue: Any, MetadataPath: Path) -> Optional[Path]:
    if PathValue is None:
        return None
    if isinstance(PathValue, float) and pd.isna(PathValue):
        return None

    PathObj = Path(str(PathValue))
    if PathObj.is_absolute() and PathObj.exists():
        return PathObj
    if PathObj.exists():
        return PathObj

    CandidatePath = MetadataPath.parent / PathObj
    if CandidatePath.exists():
        return CandidatePath

    return PathObj

# Light Curve Loading

In [3]:
def LoadLightCurve(LightCurvePath: Path) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """Load a light curve while treating flux uncertainty as optional.

    Time and flux determine whether a cadence is scientifically usable.
    FluxErr is retained only when it is aligned with the loaded arrays and all
    retained uncertainty values are finite and strictly positive. Invalid or
    missing uncertainty therefore downgrades Lomb-Scargle from weighted to
    unweighted analysis instead of removing otherwise valid QLP cadences.
    """

    def FirstAvailableColumn(ColumnNames: list[str], CandidateNames: list[str]) -> Optional[str]:
        NameMap = {Name.upper(): Name for Name in ColumnNames}
        for CandidateName in CandidateNames:
            MatchedName = NameMap.get(CandidateName.upper())
            if MatchedName is not None:
                return MatchedName
        return None

    def ReadGenericFitsTable(FitsPath: Path) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
        with fits.open(str(FitsPath), memmap=False) as Hdul:
            TableData = None

            for Hdu in Hdul:
                Data = getattr(Hdu, "data", None)
                if Data is not None and getattr(Data, "dtype", None) is not None and Data.dtype.names:
                    TableData = Data
                    break

            if TableData is None:
                raise ValueError("No FITS binary table with named columns was found")

            ColumnNames = list(TableData.dtype.names)

            TimeColumn = FirstAvailableColumn(
                ColumnNames,
                ["TIME", "TMID", "BJD"],
            )
            FluxColumn = FirstAvailableColumn(
                ColumnNames,
                ["FLUX", "SAP_FLUX", "PDCSAP_FLUX", "KSPSAP_FLUX", "DET_FLUX", "NORM_FLUX"],
            )
            FluxErrColumn = FirstAvailableColumn(
                ColumnNames,
                ["FLUX_ERR", "SAP_FLUX_ERR", "PDCSAP_FLUX_ERR", "KSPSAP_FLUX_ERR", "ERR_FLUX"],
            )

            if TimeColumn is None or FluxColumn is None:
                raise ValueError(
                    f"Required time/flux columns are missing. Available columns: {ColumnNames}"
                )

            Time = np.asarray(TableData[TimeColumn], dtype=float).reshape(-1)
            Flux = np.asarray(TableData[FluxColumn], dtype=float).reshape(-1)
            FluxErr = None

            if FluxErrColumn is not None:
                FluxErr = np.asarray(TableData[FluxErrColumn], dtype=float).reshape(-1)

            return Time, Flux, FluxErr

    def ExtractFromLightkurve(LightCurve) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
        Time = np.asarray(
            getattr(LightCurve.time, "value", LightCurve.time),
            dtype=float,
        ).reshape(-1)

        Flux = np.asarray(
            getattr(LightCurve.flux, "value", LightCurve.flux),
            dtype=float,
        ).reshape(-1)

        FluxErr = None
        if hasattr(LightCurve, "flux_err") and LightCurve.flux_err is not None:
            FluxErr = np.asarray(
                getattr(LightCurve.flux_err, "value", LightCurve.flux_err),
                dtype=float,
            ).reshape(-1)

        return Time, Flux, FluxErr

    LightCurvePath = Path(LightCurvePath)

    if not LightCurvePath.exists():
        raise FileNotFoundError(str(LightCurvePath))

    Time = None
    Flux = None
    FluxErr = None
    ReadErrors = []

    # Attempt 1: standard lightkurve reader.
    try:
        LightCurve = lk.read(str(LightCurvePath))
        Time, Flux, FluxErr = ExtractFromLightkurve(LightCurve)
    except Exception as Exc:
        ReadErrors.append(f"generic lk.read: {Exc!r}")

    # Attempt 2: QLP-specific reader.
    if Time is None or Flux is None:
        try:
            LightCurve = lk.read(
                str(LightCurvePath),
                author="QLP",
                flux_column="kspsap_flux",
            )
            Time, Flux, FluxErr = ExtractFromLightkurve(LightCurve)
        except Exception as Exc:
            ReadErrors.append(f"QLP lk.read: {Exc!r}")

    # Attempt 3: direct FITS table parsing.
    if Time is None or Flux is None:
        try:
            Time, Flux, FluxErr = ReadGenericFitsTable(LightCurvePath)
        except Exception as Exc:
            ReadErrors.append(f"generic FITS parsing: {Exc!r}")
            raise ValueError(
                "Unable to read light curve. " + " | ".join(ReadErrors)
            ) from Exc

    if Time.shape != Flux.shape:
        raise ValueError(
            f"Time/Flux shape mismatch: Time={Time.shape}, Flux={Flux.shape}"
        )

    # Only Time and Flux define a valid cadence.
    CadenceMask = np.isfinite(Time) & np.isfinite(Flux)

    FilteredTime = Time[CadenceMask]
    FilteredFlux = Flux[CadenceMask]
    FilteredFluxErr = None

    # FluxErr is optional. It must never remove otherwise valid Time/Flux rows.
    if FluxErr is not None:
        if FluxErr.shape == Time.shape:
            CandidateFluxErr = FluxErr[CadenceMask]
            ValidFluxErr = np.isfinite(CandidateFluxErr) & (CandidateFluxErr > 0)

            if len(CandidateFluxErr) > 0 and np.all(ValidFluxErr):
                FilteredFluxErr = CandidateFluxErr
            else:
                Logger.debug(
                    "Ignoring unusable flux_err for %s: %d/%d retained values are finite and positive",
                    LightCurvePath,
                    int(np.sum(ValidFluxErr)),
                    len(CandidateFluxErr),
                )
        else:
            Logger.debug(
                "Ignoring flux_err shape mismatch for %s: flux_err=%s, time=%s",
                LightCurvePath,
                FluxErr.shape,
                Time.shape,
            )

    # Keep samples in chronological order.
    Order = np.argsort(FilteredTime)
    FilteredTime = FilteredTime[Order]
    FilteredFlux = FilteredFlux[Order]

    if FilteredFluxErr is not None:
        FilteredFluxErr = FilteredFluxErr[Order]

    return FilteredTime, FilteredFlux, FilteredFluxErr

# Identification and Metadata Features

In [4]:
def ExtractIdentifierAndMetadataFeatures(Row: pd.Series) -> Dict[str, Any]:
    Result: Dict[str, Any] = {}

    ColumnsToPreserve = [
        "family", "VSXType", "VSXId", "Name", "ticId", "bestTicId",
        "ticDistanceArcmin", "lightCurvePath", "rawLightCurvePath",
        "trendFlag", "trendScore", "adfPValue",
    ]

    for ColumnName in ColumnsToPreserve:
        if ColumnName in Row.index:
            Result[ColumnName] = Row.get(ColumnName)

    QualityValue = Row.get("quality", Row.get("fitsQcStatus", Row.get("lightCurveQuality", np.nan)))
    ProvenanceValue = Row.get("provenance", Row.get("author", np.nan))

    QualityLabel = str(QualityValue) if not pd.isna(QualityValue) else "missing"
    ProvenanceLabel = str(ProvenanceValue) if not pd.isna(ProvenanceValue) else "missing"

    Result.update({
        "QualityLabel": QualityLabel,
        "QualityScore": QualityScoreMap.get(QualityLabel, np.nan),
        "Provenance": ProvenanceLabel,
        "ProvenanceScore": ProvenanceScoreMap.get(ProvenanceLabel, np.nan),
        "OriginalFluxMedian": SafeFloat(Row.get("fluxMedian", np.nan)),
        "OriginalFluxStd": SafeFloat(Row.get("fluxStd", np.nan)),
        "OriginalFluxSnr": SafeFloat(Row.get("fluxSnr", np.nan)),
        "LowSnr": SafeBool(Row.get("lowSNR", False)),
        "LowQualityLightCurve": SafeBool(Row.get("lowQualityLightCurve", False)),
    })
    return Result

# Data Coverage and Flux Distribution Features

In [5]:
def ExtractBasicStatisticalFeatures(Time: np.ndarray, Flux: np.ndarray) -> Dict[str, float]:
    CadenceCount = len(Flux)
    P01, P05, P10, P25, P50, P75, P90, P95, P99 = np.percentile(
        Flux, [1, 5, 10, 25, 50, 75, 90, 95, 99]
    )
    FluxStd = float(np.std(Flux))

    return {
        "CadenceCount": float(CadenceCount),
        "TimeSpanDays": float(np.max(Time) - np.min(Time)) if CadenceCount > 1 else np.nan,
        "MedianCadenceDays": float(np.median(np.diff(Time))) if CadenceCount > 2 else np.nan,
        "FluxStd": FluxStd,
        "FluxMedian": float(P50),
        "FluxMad": float(np.median(np.abs(Flux - P50))),
        "FluxP05": float(P05),
        "FluxP10": float(P10),
        "FluxP90": float(P90),
        "FluxP95": float(P95),
        "FluxIqr": float(P75 - P25),
        "FluxAmplitude": float(np.max(Flux) - np.min(Flux)),
        "FluxPercentAmplitude95To5": float(P95 - P05),
        "FluxPercentAmplitude90To10": float(P90 - P10),
        "FluxSkewness": float(skew(Flux, bias=False)) if CadenceCount >= 3 and FluxStd > Eps else np.nan,
        "FluxKurtosis": float(kurtosis(Flux, bias=False)) if CadenceCount >= 4 and FluxStd > Eps else np.nan,
    }

# Tail-Asymmetry Features

In [6]:
def ExtractTailAsymmetryFeatures(Flux: np.ndarray) -> Dict[str, float]:
    P05, P50, P95 = np.percentile(Flux, [5, 50, 95])

    TailUpper = float(P95 - P50)
    TailLower = float(P50 - P05)
    TailAsymmetry = float(TailUpper - TailLower)
    TailRatio = SafeDivide(TailUpper, TailLower + Eps)

    return {
        "TailUpper": TailUpper,
        "TailLower": TailLower,
        "TailAsymmetry": TailAsymmetry,
        "TailRatio": TailRatio,
    }

# Time-Domain Variability Features

In [7]:
def ExtractVariabilityFeatures(Time: np.ndarray, Flux: np.ndarray) -> Dict[str, float]:
    CadenceCount = len(Flux)
    if CadenceCount < 3:
        return {
            "EtaVonNeumann": np.nan,
            "MaxAbsSlope": np.nan,
            "MedianAbsSuccessiveDiff": np.nan,
            "FractionBeyond1Std": np.nan,
            "FractionBeyond2Std": np.nan,
        }

    FluxMean = float(np.mean(Flux))
    FluxStd = float(np.std(Flux))
    FluxVariance = float(np.var(Flux))
    FluxDiff = np.diff(Flux)
    TimeDiff = np.diff(Time)

    ValidTimeDiff = np.isfinite(TimeDiff) & (np.abs(TimeDiff) > Eps)
    Slopes = FluxDiff[ValidTimeDiff] / TimeDiff[ValidTimeDiff] if np.any(ValidTimeDiff) else np.array([])

    Eta = np.sum(FluxDiff ** 2) / ((CadenceCount - 1) * FluxVariance) if FluxVariance > Eps else np.nan

    return {
        "EtaVonNeumann": float(Eta) if np.isfinite(Eta) else np.nan,
        "MaxAbsSlope": float(np.max(np.abs(Slopes))) if Slopes.size else np.nan,
        "MedianAbsSuccessiveDiff": float(np.median(np.abs(FluxDiff))),
        "FractionBeyond1Std": float(np.mean(np.abs(Flux - FluxMean) > FluxStd)) if FluxStd > Eps else np.nan,
        "FractionBeyond2Std": float(np.mean(np.abs(Flux - FluxMean) > 2 * FluxStd)) if FluxStd > Eps else np.nan,
    }

# Lomb-Scargle Period Features

In [8]:
def ExtractLombScargleFeatures(Time: np.ndarray, Flux: np.ndarray, FluxErr: Optional[np.ndarray]) -> Dict[str, float]:
    Result = {
        "LsBestPeriod": np.nan,
        "LsMaxPower": np.nan,
        "LsFalseAlarmProbability": np.nan,
        "LsPeriod2": np.nan,
        "LsPeriod3": np.nan,
        "LsPower2": np.nan,
        "LsPower3": np.nan,
        "LsPowerRatio21": np.nan,
        "LsPowerRatio31": np.nan,
        "LsPeriodRatio21": np.nan,
        "LsPeriodRatio31": np.nan,
    }

    if len(Flux) < MinCadences:
        return Result

    TimeSpan = float(np.max(Time) - np.min(Time))
    if not np.isfinite(TimeSpan) or TimeSpan <= 0:
        return Result

    MaxPeriod = min(MaxPeriodDays, 0.9 * TimeSpan)
    if MaxPeriod <= MinPeriodDays:
        return Result

    CenteredFlux = Flux - np.nanmedian(Flux)

    try:
        if FluxErr is not None and len(FluxErr) == len(Flux) and np.all(np.isfinite(FluxErr)):
            LombScargleModel = LombScargle(Time, CenteredFlux, dy=FluxErr)
        else:
            LombScargleModel = LombScargle(Time, CenteredFlux)

        Frequency, Power = LombScargleModel.autopower(
            minimum_frequency=1.0 / MaxPeriod,
            maximum_frequency=1.0 / MinPeriodDays,
            samples_per_peak=SamplesPerPeak,
        )

        FiniteMask = np.isfinite(Frequency) & np.isfinite(Power) & (Frequency > 0)
        Frequency = Frequency[FiniteMask]
        Power = Power[FiniteMask]
        if len(Power) == 0:
            return Result

        Order = np.argsort(Power)[::-1]
        BestFrequency = float(Frequency[Order[0]])
        BestPeriod = float(1.0 / BestFrequency)
        BestPower = float(Power[Order[0]])

        Result.update({
            "LsBestPeriod": BestPeriod,
            "LsMaxPower": BestPower,
            "LsFalseAlarmProbability": float(LombScargleModel.false_alarm_probability(BestPower)),
        })

        if len(Order) > 1:
            Period2 = float(1.0 / Frequency[Order[1]])
            Power2 = float(Power[Order[1]])
            Result.update({
                "LsPeriod2": Period2,
                "LsPower2": Power2,
                "LsPowerRatio21": SafeDivide(Power2, BestPower),
                "LsPeriodRatio21": SafeDivide(Period2, BestPeriod),
            })

        if len(Order) > 2:
            Period3 = float(1.0 / Frequency[Order[2]])
            Power3 = float(Power[Order[2]])
            Result.update({
                "LsPeriod3": Period3,
                "LsPower3": Power3,
                "LsPowerRatio31": SafeDivide(Power3, BestPower),
                "LsPeriodRatio31": SafeDivide(Period3, BestPeriod),
            })

    except Exception as Exc:
        Logger.debug("Lomb-Scargle failed: %s", Exc)

    return Result

# Phase-Folded Morphology Features

In [9]:
def ExtractPhaseFeatures(Time: np.ndarray, Flux: np.ndarray, Period: float) -> Dict[str, float]:
    Result = {
        "PhaseCurveStd": np.nan,
        "PhaseCurveRange": np.nan,
        "PhaseCurveSmoothness": np.nan,
        "PhasePeakPhase": np.nan,
        "PhaseTroughPhase": np.nan,
        "PhasePeakToTroughPhaseDelta": np.nan,
    }

    if not np.isfinite(Period) or Period <= 0 or len(Flux) < MinCadences:
        return Result

    try:
        Phase = (Time % Period) / Period
        Order = np.argsort(Phase)
        Phase = Phase[Order]
        Flux = Flux[Order]

        BinEdges = np.linspace(0.0, 1.0, PhaseBinCount + 1)
        BinIndex = np.digitize(Phase, BinEdges) - 1

        BinnedPhase = []
        BinnedFlux = []

        for BinNumber in range(PhaseBinCount):
            BinMask = BinIndex == BinNumber
            if np.any(BinMask):
                BinnedPhase.append(float(np.median(Phase[BinMask])))
                BinnedFlux.append(float(np.median(Flux[BinMask])))

        BinnedPhase = np.asarray(BinnedPhase, dtype=float)
        BinnedFlux = np.asarray(BinnedFlux, dtype=float)
        if len(BinnedFlux) < 5:
            return Result

        PeakIndex = int(np.argmax(BinnedFlux))
        TroughIndex = int(np.argmin(BinnedFlux))
        PeakPhase = float(BinnedPhase[PeakIndex])
        TroughPhase = float(BinnedPhase[TroughIndex])

        RawDelta = abs(PeakPhase - TroughPhase)
        CyclicDelta = min(RawDelta, 1.0 - RawDelta)
        CyclicDiff = np.diff(np.r_[BinnedFlux, BinnedFlux[0]])

        Result.update({
            "PhaseCurveStd": float(np.std(BinnedFlux)),
            "PhaseCurveRange": float(np.max(BinnedFlux) - np.min(BinnedFlux)),
            "PhaseCurveSmoothness": float(np.std(CyclicDiff)),
            "PhasePeakPhase": PeakPhase,
            "PhaseTroughPhase": TroughPhase,
            "PhasePeakToTroughPhaseDelta": float(CyclicDelta),
        })

    except Exception as Exc:
        Logger.debug("Phase feature extraction failed: %s", Exc)

    return Result

# Per-Star Feature Extraction

In [10]:
def ExtractFeaturesForRow(Row: pd.Series, MetadataPath: Path) -> Dict[str, Any]:
    fitsBasePath = Path("/data/projects/TESS-research/data_pipeline")
    Result: Dict[str, Any] = {
        "FeatureStatus": "unknown",
        "FeatureError": None,
    }

    Result.update(ExtractIdentifierAndMetadataFeatures(Row))
    # LightCurvePath = ResolvePath(Row.get("lightCurvePath"), MetadataPath)
    LightCurvePath = Path(fitsBasePath / Row.get("lightCurvePath"))

    if LightCurvePath is None:
        Result["FeatureStatus"] = "missing_lightcurve_path"
        Result["FeatureError"] = "No lightCurvePath"
        return Result

    if not LightCurvePath.exists():
        Result["FeatureStatus"] = "missing_lightcurve_file"
        Result["FeatureError"] = str(LightCurvePath)
        return Result

    try:
        Time, Flux, FluxErr = LoadLightCurve(LightCurvePath)

        if len(Flux) < MinCadences:
            Result["FeatureStatus"] = "too_few_cadences"
            Result["FeatureError"] = f"Only {len(Flux)} finite cadences"
            Result["CadenceCount"] = float(len(Flux))
            return Result

        Result.update(ExtractBasicStatisticalFeatures(Time, Flux))
        Result.update(ExtractTailAsymmetryFeatures(Flux))
        Result.update(ExtractVariabilityFeatures(Time, Flux))

        LombScargleFeatures = ExtractLombScargleFeatures(Time, Flux, FluxErr)
        Result.update(LombScargleFeatures)

        BestPeriod = LombScargleFeatures.get("LsBestPeriod", np.nan)
        Result.update(ExtractPhaseFeatures(Time, Flux, BestPeriod))

        Result["FeatureStatus"] = "ok"

    except Exception as Exc:
        Result["FeatureStatus"] = "failed"
        Result["FeatureError"] = repr(Exc)

    return Result

# Dataset-Level Feature Extraction with Parallel Processing

In [11]:
def FinalizeFeatureDf(FeatureDf: pd.DataFrame) -> pd.DataFrame:
    if "_InputOrder" in FeatureDf.columns:
        FeatureDf = FeatureDf.sort_values("_InputOrder").drop(columns=["_InputOrder"]).reset_index(drop=True)
    return FeatureDf


def ExtractFeaturesSerial(MetadataDf: pd.DataFrame, MetadataPath: Path) -> pd.DataFrame:
    ResultRows = []
    for Position, (_, Row) in enumerate(MetadataDf.iterrows()):
        if Position % 100 == 0:
            Logger.info("Processing %s/%s", Position, len(MetadataDf))
        FeatureRow = ExtractFeaturesForRow(Row, MetadataPath)
        FeatureRow["_InputOrder"] = Position
        ResultRows.append(FeatureRow)
    return FinalizeFeatureDf(pd.DataFrame(ResultRows))


def ExtractFeaturesParallel(MetadataDf: pd.DataFrame, MetadataPath: Path, WorkerCount: int = WorkerCount) -> pd.DataFrame:
    ResultRows = []
    TotalRows = len(MetadataDf)
    Logger.info("Starting parallel feature extraction with WorkerCount=%s", WorkerCount)

    with ThreadPoolExecutor(max_workers=WorkerCount) as Executor:
        FutureMap = {}
        for Position, (_, Row) in enumerate(MetadataDf.iterrows()):
            Future = Executor.submit(ExtractFeaturesForRow, Row, MetadataPath)
            FutureMap[Future] = Position

        CompletedCount = 0
        for Future in as_completed(FutureMap):
            Position = FutureMap[Future]
            try:
                FeatureRow = Future.result()
            except Exception as Exc:
                FeatureRow = {"FeatureStatus": "failed", "FeatureError": repr(Exc)}

            FeatureRow["_InputOrder"] = Position
            ResultRows.append(FeatureRow)
            CompletedCount += 1

            if CompletedCount % 100 == 0 or CompletedCount == TotalRows:
                Logger.info("Completed %s/%s", CompletedCount, TotalRows)

    return FinalizeFeatureDf(pd.DataFrame(ResultRows))


def ExtractFeatures(MetadataDf: pd.DataFrame, MetadataPath: Path, WorkerCount: int = WorkerCount) -> pd.DataFrame:
    if WorkerCount <= 1:
        return ExtractFeaturesSerial(MetadataDf, MetadataPath)
    return ExtractFeaturesParallel(MetadataDf, MetadataPath, WorkerCount=WorkerCount)

# Execute Feature Extraction

In [ ]:
FeatureDfByCase = {}

for CaseName, Paths in FeatureExtractionCases.items():
    print("=" * 100)
    print(f"Running feature extraction case: {CaseName}")
    print(f"Input:  {Paths['InputMetadataPath']}")
    print(f"Output: {Paths['OutputFeaturePath']}")
    print("=" * 100)

    InputMetadataPath = Paths["InputMetadataPath"]
    OutputFeaturePath = Paths["OutputFeaturePath"]

    MetadataDf = pd.read_parquet(InputMetadataPath)

    FeatureDf = ExtractFeatures(
        MetadataDf=MetadataDf,
        MetadataPath=InputMetadataPath,
        WorkerCount=WorkerCount,
    )

    OutputFeaturePath.parent.mkdir(parents=True, exist_ok=True)
    FeatureDf.to_parquet(OutputFeaturePath, index=False)

    print(f"Saved {len(FeatureDf)} rows and {len(FeatureDf.columns)} columns to {OutputFeaturePath}")
    FeatureDf["FeatureStatus"].value_counts(dropna=False)

    FeatureDfByCase[CaseName] = FeatureDf.copy()


2026-08-06 16:51:42,493 INFO:FeatureExtractorNotebook:Starting parallel feature extraction with WorkerCount=16


Running feature extraction case: NoDetrend
Input:  /data/projects/TESS-research/data_pipeline/TESSCache/TESSAugmented_QC.parquet
Output: /data/projects/TESS-research/feature_extraction/TESS_features.parquet


2026-08-06 16:51:42,687 WARNING:astropy:UnitsWarning: 'btjd' did not parse as fits unit: At col 0, Unit 'btjd' not supported by the FITS standard.  If this is meant to be a custom unit, define it with 'u.def_unit'. To have it recognized inside a file reader or other code, enable it with 'u.add_enabled_units'. For details, see https://docs.astropy.org/en/latest/units/combining_and_defining.html
0% (0/28154) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-08-06 16:51:42,761 INFO:lightkurve.utils:0% (0/28154) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/12767) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
2026-08-06 16:51:42,791 INFO:lightkurve.utils:0% (0/12767) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/28638) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
0% (0/18678) of the cadences will be igno

# Quick Sanity Checks

In [ ]:
display(FeatureDf.head())

NumericColumns = FeatureDf.select_dtypes(include=[np.number]).columns
MissingRate = FeatureDf[NumericColumns].isna().mean().sort_values(ascending=False)
display(MissingRate.head(30))

,FeatureStatus,FeatureError,family,VSXType,VSXId,lightCurvePath,rawLightCurvePath,QualityLabel,QualityScore,Provenance,...,LsPowerRatio21,LsPowerRatio31,LsPeriodRatio21,LsPeriodRatio31,PhaseCurveStd,PhaseCurveRange,PhaseCurveSmoothness,PhasePeakPhase,PhaseTroughPhase,PhasePeakToTroughPhaseDelta
0,ok,NaN,CEPHEID,DCEPS,Gaia DR3 4685823446113073792,/data/projects/TESS-research/data_pipeline/TES...,/data/projects/TESS-research/data_pipeline/TES...,clean,0.0,TESSCut,...,0.965596,0.962654,1.000523,0.999478,0.108526,0.363780,0.089044,0.625510,0.974632,0.349122
1,ok,NaN,CEPHEID,CWB,ASASSN-V J001916.11-180430.8,/data/projects/TESS-research/data_pipeline/TES...,/data/projects/TESS-research/data_pipeline/TES...,clean,0.0,TESSCut,...,0.991554,0.989106,1.012854,0.987469,0.160734,0.640752,0.156908,0.787201,0.077085,0.289884
2,ok,NaN,CEPHEID,CWB,CSS_J014935.9+141132,/data/projects/TESS-research/data_pipeline/TES...,/data/projects/TESS-research/data_pipeline/TES...,clean,0.0,TESSCut,...,0.988269,0.981074,1.019496,0.981236,0.346630,1.472735,0.332614,0.026434,0.275024,0.248590
3,ok,NaN,CEPHEID,DCEP,Gaia DR3 461049865760507904,/data/projects/TESS-research/data_pipeline/TES...,/data/projects/TESS-research/data_pipeline/TES...,clean,0.0,QLP,...,0.977464,0.976398,0.999790,1.000210,0.073633,0.217096,0.025917,0.775062,0.475095,0.299967
4,ok,NaN,CEPHEID,DCEPS,Gaia DR3 4662956971490538880,/data/projects/TESS-research/data_pipeline/TES...,/data/projects/TESS-research/data_pipeline/TES...,clean,0.0,TESSCut,...,0.990783,0.915923,1.000500,0.999501,0.034486,0.119370,0.021449,0.675007,0.325098,0.349909


TimeSpanDays                  0.001782
FluxP10                       0.001782
FluxP05                       0.001782
FluxMad                       0.001782
FluxMedian                    0.001782
FluxStd                       0.001782
MedianCadenceDays             0.001782
FluxP90                       0.001782
FluxP95                       0.001782
FluxKurtosis                  0.001782
FluxSkewness                  0.001782
FluxPercentAmplitude90To10    0.001782
FluxPercentAmplitude95To5     0.001782
FluxAmplitude                 0.001782
FluxIqr                       0.001782
LsMaxPower                    0.001782
LsFalseAlarmProbability       0.001782
LsPeriod2                     0.001782
LsPeriod3                     0.001782
LsPower2                      0.001782
LsPower3                      0.001782
TailUpper                     0.001782
TailLower                     0.001782
TailAsymmetry                 0.001782
TailRatio                     0.001782
EtaVonNeumann            

In [ ]:
if "family" in FeatureDf.columns:
    display(pd.crosstab(FeatureDf["family"], FeatureDf["FeatureStatus"], dropna=False))

FeatureStatus,failed,ok,too_few_cadences
family,,,
CEPHEID,0,962,1
CV,0,516,0
DSCT_SXPHE,0,968,0
ECLIPSING,0,956,0
ELLIPSOIDAL_ROT,0,982,0
LONG_PERIOD,0,959,0
RRLYR,0,917,0
XRAY,0,51,0
YSO,0,970,1


## Consolidated Output Summary

Display the paths, dimensions, and feature-extraction status counts for both cases.


In [ ]:
SummaryRows = []

for CaseName, Paths in FeatureExtractionCases.items():
    CaseFeatureDf = FeatureDfByCase.get(CaseName)

    if CaseFeatureDf is None:
        print(f"{CaseName}: run the feature-extraction cell first.")
        continue

    SummaryRows.append({
        "Case": CaseName,
        "InputMetadataPath": str(Paths["InputMetadataPath"]),
        "OutputFeaturePath": str(Paths["OutputFeaturePath"]),
        "Rows": len(CaseFeatureDf),
        "Columns": len(CaseFeatureDf.columns),
    })

    print("=" * 100)
    print(CaseName)
    print("=" * 100)
    print("InputMetadataPath:", Paths["InputMetadataPath"])
    print("OutputFeaturePath:", Paths["OutputFeaturePath"])
    print("Shape:", CaseFeatureDf.shape)

    if "FeatureStatus" in CaseFeatureDf.columns:
        display(
            CaseFeatureDf["FeatureStatus"]
            .value_counts(dropna=False)
            .rename_axis("FeatureStatus")
            .reset_index(name="Count")
        )

if SummaryRows:
    display(pd.DataFrame(SummaryRows))
